# UK Used Car Price Prediction: 02. Exploratory Data Analysis (EDA)

Notebook ini mengeksplorasi data yang sudah dibersihkan di notebook 01 untuk memahami variabel target (`price`) dan hubungannya dengan fitur-fitur lain, sebelum masuk ke pemodelan.

**Input:** `data/processed/cars_clean.csv` (hasil notebook 01, jalankan notebook 01 dulu).
**Output:** grafik di `reports/figures/` dan catatan temuan di bagian akhir.

**Catatan:** EDA hanya untuk memahami data. Tidak ada keputusan yang memakai test set di sini, karena split train/test dilakukan di notebook berikutnya.

## 1. Setup dan Load Data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA = Path("../data/processed/cars_clean.csv")
FIG = Path("../reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
print(df.shape)
df.head()

In [ ]:
# Fitur umur mobil. Asumsi: data diambil sekitar pertengahan 2020.
REF_YEAR = 2020
df["car_age"] = REF_YEAR - df["year"]

print(df["car_age"].describe().round(2))
print("\nJumlah baris per merek:")
print(df["brand"].value_counts().to_string())

## 2. Distribusi Harga (Target)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(df["price"], bins=60)
axes[0].set_title("Distribusi harga (£)")
axes[0].set_xlabel("price")

axes[1].hist(np.log(df["price"]), bins=60)
axes[1].set_title("Distribusi log(harga)")
axes[1].set_xlabel("log(price)")

plt.tight_layout()
plt.savefig(FIG / "eda_price_distribution.png", dpi=150)
plt.show()

print("skewness harga      :", round(df["price"].skew(), 2))
print("skewness log(harga) :", round(np.log(df["price"]).skew(), 2))
print()
print(df["price"].describe().round(0).to_string())

## 3. Harga vs Umur dan Mileage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot harga per umur (dibatasi <= 20 tahun agar terbaca; outlier disembunyikan)
sub = df[df["car_age"] <= 20]
sns.boxplot(data=sub, x="car_age", y="price", showfliers=False, ax=axes[0])
axes[0].set_title("Harga vs umur mobil (tahun)")

# Scatter mileage vs harga (sampel agar ringan)
samp = df.sample(min(20000, len(df)), random_state=42)
axes[1].scatter(samp["mileage"], samp["price"], s=3, alpha=0.2)
axes[1].set_title("Harga vs mileage (sampel)")
axes[1].set_xlabel("mileage")
axes[1].set_ylabel("price")

plt.tight_layout()
plt.savefig(FIG / "eda_price_vs_age_mileage.png", dpi=150)
plt.show()

print("Korelasi Spearman dengan harga:")
print("  car_age :", round(df["price"].corr(df["car_age"], method="spearman"), 3))
print("  mileage :", round(df["price"].corr(df["mileage"], method="spearman"), 3))
print("Korelasi Spearman umur vs mileage:",
      round(df["car_age"].corr(df["mileage"], method="spearman"), 3))

## 4. Harga per Merek

In [ ]:
order = df.groupby("brand")["price"].median().sort_values(ascending=False).index

plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="brand", y="price", order=order, showfliers=False)
plt.title("Distribusi harga per merek")
plt.tight_layout()
plt.savefig(FIG / "eda_price_by_brand.png", dpi=150)
plt.show()

summary = (df.groupby("brand")["price"]
             .agg(jumlah="count", median="median", rata_rata="mean")
             .round(0)
             .loc[order])
summary

## 5. Transmission dan Fuel Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=df, x="transmission", y="price", showfliers=False, ax=axes[0])
axes[0].set_title("Harga per transmission")

sns.boxplot(data=df, x="fuelType", y="price", showfliers=False, ax=axes[1])
axes[1].set_title("Harga per fuel type")

plt.tight_layout()
plt.savefig(FIG / "eda_price_by_categories.png", dpi=150)
plt.show()

# Perhatikan jumlah baris tiap kategori: kategori kecil kurang bisa diandalkan
for col in ["transmission", "fuelType"]:
    print(df.groupby(col)["price"].agg(jumlah="count", median="median").round(0), "\n")

## 6. Engine Size, MPG, dan Tax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col in zip(axes, ["engineSize", "mpg", "tax"]):
    ax.scatter(samp[col], samp["price"], s=3, alpha=0.2)
    ax.set_title(f"Harga vs {col} (sampel)")
    ax.set_xlabel(col)
    ax.set_ylabel("price")

plt.tight_layout()
plt.savefig(FIG / "eda_price_vs_engine_mpg_tax.png", dpi=150)
plt.show()

print("Jumlah baris mpg > 100 (plug-in hybrid, dipertahankan di cleaning):",
      (df["mpg"] > 100).sum())
print("Jumlah baris tax == 0:", (df["tax"] == 0).sum())

## 7. Korelasi Antar Fitur Numerik

In [ ]:
num_cols = ["price", "car_age", "mileage", "tax", "mpg", "engineSize"]
corr = df[num_cols].corr(method="spearman")

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Korelasi Spearman")
plt.tight_layout()
plt.savefig(FIG / "eda_correlation_heatmap.png", dpi=150)
plt.show()

print(corr["price"].drop("price").sort_values(key=abs, ascending=False).round(3).to_string())

## 8. Temuan EDA

Isi bagian ini setelah melihat grafik dan angka di atas, dengan kata-katamu sendiri dan sertakan angkanya.

- **Distribusi harga:** [bentuk distribusi? skewness harga vs log(harga)? apa implikasinya untuk transformasi target?]
- **Harga vs umur dan mileage:** [arah dan kekuatan hubungan; apakah umur dan mileage saling berkorelasi kuat?]
- **Harga per merek:** [merek dengan harga median tertinggi dan terendah; sebaran di dalam merek]
- **Transmission dan fuel type:** [kategori mana yang lebih mahal; kategori mana yang jumlahnya terlalu kecil?]
- **Engine size, mpg, tax:** [hubungan dengan harga; apakah ada pola non-linear?]
- **Korelasi antar fitur:** [pasangan fitur yang berkorelasi kuat dan dampaknya untuk model linear]

### Implikasi untuk pemodelan
- [transformasi target (log) atau tidak, beserta alasannya]
- [fitur turunan yang dipakai: `car_age`]
- [penanganan kolom `model` yang kategorinya banyak: `OneHotEncoder(min_frequency=10)`]
- [hal lain yang perlu diperhatikan dari grafik]